# Notebook 1A: TF-IDF + MiniBatchKMeans Baseline

This is a lightweight lexical baseline for the Milestone 3 method comparison. It does **not**
replace the proposed MiniLM + UMAP + HDBSCAN + BERTopic discovery pipeline. Its purpose is to test
whether a simpler bag-of-words method can recover balanced and coherent scam archetypes.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA = PROJECT_ROOT / "data/processed/complaints_clean.csv.gz"
OUT = PROJECT_ROOT / "data/processed"
SEED = 42
SAMPLE_SIZE = 3000
K = 8

df = pd.read_csv(DATA)
work = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=SEED).copy().reset_index(drop=True)
print(f"Fixed development sample: {len(work):,} rows")

Fixed development sample: 3,000 rows


In [2]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=4,
    max_df=0.90,
    max_features=10_000,
    sublinear_tf=True,
    norm="l2",
)
X = vectorizer.fit_transform(work["clean_text"].fillna(""))
print("TF-IDF matrix:", X.shape)

model = MiniBatchKMeans(
    n_clusters=K,
    random_state=SEED,
    batch_size=256,
    n_init=1,
    max_iter=15,
    max_no_improvement=3,
)
labels = model.fit_predict(X)
work["baseline_topic"] = labels
print(work["baseline_topic"].value_counts().sort_index())

TF-IDF matrix: (3000, 9841)


baseline_topic
0      1
1    432
2    877
3    786
4    118
5     76
6    510
7    200
Name: count, dtype: int64


In [3]:
rng = np.random.RandomState(SEED)
sample_idx = rng.choice(len(work), size=min(800, len(work)), replace=False)
silhouette = silhouette_score(X[sample_idx], labels[sample_idx], metric="cosine")
counts = work["baseline_topic"].value_counts()
largest_share = counts.max() / len(work)
print(f"Cosine silhouette on fixed evaluation sample: {silhouette:.3f}")
print(f"Largest-cluster share: {largest_share:.1%}")
print(f"Smallest cluster: {counts.min():,} record(s)")

Cosine silhouette on fixed evaluation sample: 0.012
Largest-cluster share: 29.2%
Smallest cluster: 1 record(s)


In [4]:
terms = np.asarray(vectorizer.get_feature_names_out())
review_rows = []
for topic_id in range(K):
    member_idx = np.where(labels == topic_id)[0]
    top_term_idx = model.cluster_centers_[topic_id].argsort()[::-1][:15]
    keywords = ", ".join(terms[top_term_idx])

    similarities = np.asarray(X[member_idx].dot(model.cluster_centers_[topic_id])).reshape(-1)
    representative_local_idx = np.argsort(similarities)[::-1][:3]
    representatives = [
        work.loc[member_idx[i], "clean_text"][:600].replace("\n", " ")
        for i in representative_local_idx
    ]
    review_rows.append({
        "topic_id": topic_id,
        "count": len(member_idx),
        "keywords": keywords,
        "representative_1": representatives[0] if len(representatives) > 0 else "",
        "representative_2": representatives[1] if len(representatives) > 1 else "",
        "representative_3": representatives[2] if len(representatives) > 2 else "",
    })

review = pd.DataFrame(review_rows).sort_values("count", ascending=False)
review[["topic_id", "count", "keywords"]]

,topic_id,count,keywords
2,2,877,"redacted redacted, account, crypto_exchange, b..."
3,3,786,"redacted redacted, bank, money, money_amount, ..."
6,6,510,"payment_app, redacted redacted, money, sent, m..."
1,1,432,"redacted redacted, payment_app, bank, transact..."
7,7,200,"wire, bank, wire transfer, redacted redacted, ..."
4,4,118,"credit, redacted redacted, credit report, debt..."
5,5,76,"fell victim, fell, commencing redacted, commen..."
0,0,1,"closed investigation, transactions taken, chec..."


In [5]:
review.to_csv(OUT / "tfidf_kmeans_topic_review_sample.csv", index=False)
work.to_csv(OUT / "tfidf_kmeans_sample_output.csv.gz", index=False, compression="gzip")
summary = {
    "sample_size": int(len(work)),
    "k": int(K),
    "silhouette_cosine": float(silhouette),
    "largest_cluster_share": float(largest_share),
    "smallest_cluster_size": int(counts.min()),
    "selected_for_final_pipeline": False,
}
with open(OUT / "tfidf_kmeans_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
summary

{'sample_size': 3000,
 'k': 8,
 'silhouette_cosine': 0.012206057330885011,
 'largest_cluster_share': 0.29233333333333333,
 'smallest_cluster_size': 1,
 'selected_for_final_pipeline': False}

## Decision

This baseline is rejected as the final discovery method. The cosine silhouette is very low and one
cluster collapses to a single complaint, indicating unstable lexical separation. The result supports
continuing with the proposal's semantic BERTopic approach while retaining this notebook as a
transparent benchmark.